In [17]:
# ============================================================
# CELL 1 — KAGGLE SETUP + LOAD GOVERNMENT TRAIN/TEST DATA
# Model experiment: meta-llama/Llama-3.1-8B-Instruct
#
# Kaggle input:
#   government_chat_train.jsonl
#   merged_test_data.csv
#
# Creates:
#   unified_train.json
#   few_shot_examples.json
#   test_questions_used.csv
#
# Uses fixed 3-shot examples per domain
# ============================================================

!pip install -q "transformers==4.48.3" accelerate bitsandbytes \
    "huggingface_hub<1.0" sacrebleu rapidfuzz bert-score==0.3.13

import os
import json
import random
import re
import gc
import unicodedata

from pathlib import Path

import pandas as pd
import numpy as np
import torch


# ============================================================
# 1. PATHS
# ============================================================

INPUT_ROOT = Path("/kaggle/input")

WORK_ROOT = Path(
    "/kaggle/working/BanglaLLM"
)

RAG_DIR = (
    WORK_ROOT
    / "RAG"
)

OUT = (
    WORK_ROOT
    / "Llama"
    / "Few-Shot"
)

RAG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


print("=" * 70)
print("PATH CONFIGURATION")
print("=" * 70)

print("Kaggle input root:")
print(INPUT_ROOT)

print("\nRAG output:")
print(RAG_DIR)

print("\nLlama few-shot output:")
print(OUT)


# ============================================================
# 2. FIND EXACT KAGGLE INPUT FILES
# ============================================================

def find_exact_file(filename):

    matches = list(
        INPUT_ROOT.rglob(filename)
    )

    matches = [
        p
        for p in matches
        if p.is_file()
    ]

    if not matches:

        print("\nAvailable files under /kaggle/input:")

        for p in INPUT_ROOT.rglob("*"):

            if p.is_file():
                print(" ", p)

        raise FileNotFoundError(
            f"\nCould not find:\n{filename}\n"
        )

    if len(matches) > 1:

        print(
            f"\nMultiple copies of {filename} found:"
        )

        for p in matches:
            print(" ", p)

    matches = sorted(
        matches,
        key=lambda x: (
            len(str(x)),
            str(x)
        )
    )

    return matches[0]


TRAIN_PATH = find_exact_file(
    "government_chat_train.jsonl"
)

TEST_PATH = find_exact_file(
    "merged_test_data.csv"
)


print("\n" + "=" * 70)
print("INPUT FILES FOUND")
print("=" * 70)

print("Train:")
print(TRAIN_PATH)

print("\nTest:")
print(TEST_PATH)


# ============================================================
# 3. DOMAIN NORMALIZATION
# ============================================================

def norm_domain(value):

    value = str(
        value
    ).lower().strip()

    value = value.replace(
        "-",
        "_"
    )

    value = value.replace(
        " ",
        "_"
    )

    # Passport
    if "passport" in value:
        return "passport"

    # NID
    if (
        "nid" in value
        or "national_id" in value
        or "national_identity" in value
    ):
        return "nid"

    # TIN
    if (
        value == "tin"
        or "tax" in value
        or "taxpayer" in value
        or "etin" in value
        or "e_tin" in value
    ):
        return "tin"

    # Birth / Death
    if (
        "birth" in value
        or "death" in value
    ):
        return "birth_death"

    return value


# ============================================================
# 4. SAFE TEXT HELPER
# ============================================================

def clean_text(value):

    if value is None:
        return ""

    # Protect against pandas/JSON NaN values
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    return str(value).strip()


# ============================================================
# 5. BUILD QUESTION
#
# IMPORTANT:
#
# Some rows have:
#
#   instruction = actual Bangla question
#   input       = empty
#
# Other datasets may use input instead.
#
# Therefore we use row-by-row fallback.
# ============================================================

def build_question(row):

    question = clean_text(
        row.get(
            "question",
            ""
        )
    )

    input_text = clean_text(
        row.get(
            "input",
            ""
        )
    )

    instruction = clean_text(
        row.get(
            "instruction",
            ""
        )
    )

    # Explicit question has highest priority
    if question:
        return question

    # Use input only when actually non-empty
    if input_text:
        return input_text

    # Current merged_test_data.csv uses this
    if instruction:
        return instruction

    return ""


# ============================================================
# 6. BUILD ANSWER
# ============================================================

def build_answer(row):

    for column in [
        "output",
        "answer",
        "gold",
        "response",
    ]:

        value = clean_text(
            row.get(
                column,
                ""
            )
        )

        if value:
            return value

    return ""


# ============================================================
# 7. LOAD JSONL TRAIN DATA
# ============================================================

print("\n" + "=" * 70)
print("LOADING TRAIN DATA")
print("=" * 70)


train_rows = []


with open(
    TRAIN_PATH,
    "r",
    encoding="utf-8"
) as f:

    for line_number, line in enumerate(
        f,
        start=1
    ):

        line = line.strip()

        if not line:
            continue

        try:

            row = json.loads(
                line
            )

            train_rows.append(
                row
            )

        except json.JSONDecodeError as e:

            raise ValueError(
                f"Invalid JSON on line "
                f"{line_number} of:\n"
                f"{TRAIN_PATH}\n\n"
                f"{e}"
            )


print(
    "Raw train rows:",
    len(train_rows)
)


if len(train_rows) == 0:

    raise RuntimeError(
        "government_chat_train.jsonl "
        "contains no records."
    )


# ============================================================
# 8. SHOW TRAIN SCHEMA
# ============================================================

print("\nTrain JSONL columns:")

train_columns = sorted(
    {
        key
        for row in train_rows
        for key in row.keys()
    }
)

print(
    train_columns
)


# ============================================================
# 9. BUILD CLEAN UNIFIED TRAIN SET
# ============================================================

unified_train = []

seen = set()


for index, row in enumerate(
    train_rows
):

    domain = norm_domain(
        row.get(
            "domain",
            ""
        )
    )

    question = build_question(
        row
    )

    answer = build_answer(
        row
    )


    # Skip invalid examples
    if not domain:
        continue

    if not question:
        continue

    if not answer:
        continue


    row_id = clean_text(
        row.get(
            "id",
            index
        )
    )

    if not row_id:
        row_id = str(index)


    key = (
        domain,
        question,
        answer,
    )


    if key in seen:
        continue


    seen.add(
        key
    )


    unified_train.append(
        {
            "id": row_id,

            "domain": domain,

            "topic": clean_text(
                row.get(
                    "topic",
                    ""
                )
            ),

            "question_type": clean_text(
                row.get(
                    "question_type",
                    ""
                )
            ),

            "instruction": clean_text(
                row.get(
                    "instruction",
                    ""
                )
            ),

            "input": clean_text(
                row.get(
                    "input",
                    ""
                )
            ),

            # Standardized fields
            "question": question,

            "answer": answer,

            # Keep output for compatibility
            "output": answer,

            "source_url": clean_text(
                row.get(
                    "source_url",
                    ""
                )
            ),

            "split": clean_text(
                row.get(
                    "split",
                    ""
                )
            ),

            "source": clean_text(
                row.get(
                    "source",
                    ""
                )
            ),
        }
    )


print(
    "Clean train rows:",
    len(unified_train)
)


if len(unified_train) == 0:

    raise RuntimeError(
        "No valid training examples were created."
    )


# ============================================================
# 10. TRAIN DATAFRAME + DOMAIN DISTRIBUTION
# ============================================================

train_df = pd.DataFrame(
    unified_train
)


print("\nTraining examples per domain:")

print(
    train_df[
        "domain"
    ]
    .value_counts(
        dropna=False
    )
    .to_string()
)


# ============================================================
# 11. SAVE UNIFIED TRAIN SET
# ============================================================

UNIFIED_PATH = (
    RAG_DIR
    / "unified_train.json"
)


with open(
    UNIFIED_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        unified_train,
        f,
        ensure_ascii=False,
        indent=2
    )


print(
    "\nSaved unified train:",
    UNIFIED_PATH
)


# ============================================================
# 12. LOAD TEST DATA
# ============================================================

print("\n" + "=" * 70)
print("LOADING TEST DATA")
print("=" * 70)


tests = pd.read_csv(
    TEST_PATH
).fillna("")


print(
    "Raw test rows:",
    len(tests)
)

print(
    "\nOriginal test columns:"
)

print(
    list(
        tests.columns
    )
)


# ============================================================
# 13. INSPECT WHERE TEST QUESTIONS ACTUALLY LIVE
# ============================================================

for column in [
    "question",
    "instruction",
    "input",
    "output",
]:

    if column in tests.columns:

        nonempty = (
            tests[column]
            .astype(str)
            .str.strip()
            .ne("")
            .sum()
        )

        print(
            f"Non-empty '{column}' rows:",
            nonempty
        )


# ============================================================
# 14. STANDARDIZE TEST QUESTION
#
# FIX:
# Do NOT blindly use tests["input"] merely because
# the column exists.
#
# Current dataset:
#   input       -> empty
#   instruction -> question
# ============================================================

tests["question"] = tests.apply(
    lambda row: build_question(
        row.to_dict()
    ),
    axis=1
)


# ============================================================
# 15. STANDARDIZE TEST GOLD ANSWER
# ============================================================

tests["gold"] = tests.apply(
    lambda row: build_answer(
        row.to_dict()
    ),
    axis=1
)


# ============================================================
# 16. ID COLUMN
# ============================================================

if "id" not in tests.columns:

    tests[
        "id"
    ] = [
        str(i)
        for i in range(
            len(tests)
        )
    ]


tests[
    "id"
] = tests[
    "id"
].astype(str)


# ============================================================
# 17. DOMAIN COLUMN
# ============================================================

if "domain" not in tests.columns:

    raise KeyError(
        "'domain' column is required "
        "for domain-specific few-shot inference.\n\n"
        f"Available columns:\n"
        f"{list(tests.columns)}"
    )


tests[
    "domain"
] = tests[
    "domain"
].apply(
    norm_domain
)


# ============================================================
# 18. VERIFY TEST QUESTIONS / ANSWERS
# ============================================================

empty_questions = (
    tests[
        "question"
    ]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

empty_answers = (
    tests[
        "gold"
    ]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)


print(
    "\nEmpty test questions:",
    empty_questions
)

print(
    "Empty gold answers:",
    empty_answers
)


if empty_questions > 0:

    print(
        "\nRows with empty questions:"
    )

    print(
        tests.loc[
            tests["question"]
            .astype(str)
            .str.strip()
            .eq(""),
            [
                c
                for c in [
                    "id",
                    "domain",
                    "instruction",
                    "input",
                ]
                if c in tests.columns
            ]
        ].head(20)
    )

    raise RuntimeError(
        f"{empty_questions} test rows "
        "still have empty questions."
    )


if empty_answers > 0:

    print(
        "\nRows with empty gold answers:"
    )

    print(
        tests.loc[
            tests["gold"]
            .astype(str)
            .str.strip()
            .eq(""),
            [
                c
                for c in [
                    "id",
                    "domain",
                    "question",
                    "output",
                ]
                if c in tests.columns
            ]
        ].head(20)
    )

    raise RuntimeError(
        f"{empty_answers} test rows "
        "have empty gold answers."
    )


# ============================================================
# 19. SHOW TEST SAMPLES
# ============================================================

print("\n" + "=" * 70)
print("TEST QUESTION CHECK")
print("=" * 70)


for i in range(
    min(
        5,
        len(tests)
    )
):

    print(
        f"\nRow {i}"
    )

    print(
        "ID:",
        tests.iloc[i][
            "id"
        ]
    )

    print(
        "Domain:",
        tests.iloc[i][
            "domain"
        ]
    )

    print(
        "Question:",
        tests.iloc[i][
            "question"
        ]
    )

    print(
        "Gold:",
        str(
            tests.iloc[i][
                "gold"
            ]
        )[:250]
    )


# ============================================================
# 20. TEST DOMAIN DISTRIBUTION
# ============================================================

print(
    "\nTest examples per domain:"
)

print(
    tests[
        "domain"
    ]
    .value_counts(
        dropna=False
    )
    .to_string()
)


# ============================================================
# 21. CHECK TRAIN / TEST DOMAIN COMPATIBILITY
# ============================================================

train_domains = set(
    train_df[
        "domain"
    ].unique()
)

test_domains = set(
    tests[
        "domain"
    ].unique()
)


missing_domains = (
    test_domains
    -
    train_domains
)


if missing_domains:

    raise RuntimeError(
        "Some test domains do not exist "
        "in the training data:\n"
        f"{sorted(missing_domains)}\n\n"
        "Train domains:\n"
        f"{sorted(train_domains)}"
    )


# ============================================================
# 22. BUILD FIXED 3-SHOT EXAMPLES
# ============================================================

NUM_SHOTS = 3

RANDOM_SEED = 42


random.seed(
    RANDOM_SEED
)


few_shot_examples = {}


for domain in sorted(
    test_domains
):

    domain_rows = [
        row
        for row in unified_train
        if row[
            "domain"
        ] == domain
    ]


    if len(
        domain_rows
    ) < NUM_SHOTS:

        raise ValueError(
            f"Not enough training examples "
            f"for domain '{domain}'.\n"
            f"Found: {len(domain_rows)}\n"
            f"Required: {NUM_SHOTS}"
        )


    selected = random.sample(
        domain_rows,
        NUM_SHOTS
    )


    few_shot_examples[
        domain
    ] = [
        {
            "id": row[
                "id"
            ],

            "question": row[
                "question"
            ],

            "answer": row[
                "answer"
            ],
        }

        for row in selected
    ]


# ============================================================
# 23. SAVE FIXED FEW-SHOT EXAMPLES
# ============================================================

FEW_SHOT_PATH = (
    OUT
    / "few_shot_examples.json"
)


with open(
    FEW_SHOT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        few_shot_examples,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 24. SAVE EXACT TEST SET USED
# ============================================================

TEST_USED_PATH = (
    OUT
    / "test_questions_used.csv"
)


tests.to_csv(
    TEST_USED_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 25. PRINT SELECTED SHOTS
# ============================================================

print("\n" + "=" * 70)
print("FIXED 3-SHOT EXAMPLES")
print("=" * 70)


for domain, examples in few_shot_examples.items():

    print(
        f"\nDOMAIN: {domain}"
    )

    print(
        "-" * 70
    )

    for number, example in enumerate(
        examples,
        start=1
    ):

        print(
            f"\nSHOT {number}"
        )

        print(
            "Question:",
            example[
                "question"
            ]
        )

        print(
            "Answer:",
            example[
                "answer"
            ][:250]
        )


# ============================================================
# 26. FINAL VERIFICATION
# ============================================================

print("\n" + "=" * 70)
print("CELL 1 COMPLETED SUCCESSFULLY")
print("=" * 70)


print(
    "Training examples:",
    len(unified_train)
)

print(
    "Test questions:",
    len(tests)
)

print(
    "Number of domains:",
    len(few_shot_examples)
)

print(
    "Shots per domain:",
    NUM_SHOTS
)


print("\nFiles saved:")

print(
    UNIFIED_PATH
)

print(
    FEW_SHOT_PATH
)

print(
    TEST_USED_PATH
)


print("\nRequired variables ready for CELL 2:")

print(
    "tests:",
    type(tests)
)

print(
    "few_shot_examples:",
    type(few_shot_examples)
)

print(
    "OUT:",
    OUT
)

print(
    "NUM_SHOTS:",
    NUM_SHOTS
)

PATH CONFIGURATION
Kaggle input root:
/kaggle/input

RAG output:
/kaggle/working/BanglaLLM/RAG

Llama few-shot output:
/kaggle/working/BanglaLLM/Llama/Few-Shot

INPUT FILES FOUND
Train:
/kaggle/input/datasets/akra1234/government/government_chat_train.jsonl

Test:
/kaggle/input/datasets/akra1234/government/merged_test_data.csv

LOADING TRAIN DATA
Raw train rows: 1186

Train JSONL columns:
['domain', 'id', 'input', 'instruction', 'output', 'question_type', 'source_url', 'split', 'topic']
Clean train rows: 1186

Training examples per domain:
domain
birth_death    330
passport       312
nid            307
tin            237

Saved unified train: /kaggle/working/BanglaLLM/RAG/unified_train.json

LOADING TEST DATA
Raw test rows: 248

Original test columns:
['id', 'domain', 'topic', 'question_type', 'instruction', 'input', 'output', 'source_url', 'split', 'source']
Non-empty 'instruction' rows: 248
Non-empty 'input' rows: 0
Non-empty 'output' rows: 248

Empty test questions: 0
Empty gold answ

In [22]:
# ============================================================
# CELL 2 — META LLAMA 3.1 8B INSTRUCT: 3-SHOT INFERENCE
# Kaggle + HF_TOKEN + 4-bit NF4
# No RAG, no fine-tuning
# ============================================================

from huggingface_hub import login, HfApi
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from tqdm.auto import tqdm

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

MAX_INPUT_TOKENS = 8192
MAX_NEW_TOKENS = 500
NUM_SHOTS = 3


# ============================================================
# 1. GPU CHECK
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not enabled. In Kaggle, open Notebook settings "
        "and select a GPU accelerator."
    )

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory:",
    f"{torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB"
)


# ============================================================
# 2. HUGGING FACE TOKEN FROM KAGGLE SECRETS
# Secret name must be exactly: HF_TOKEN
# ============================================================

HF_TOKEN = None

try:
    from kaggle_secrets import UserSecretsClient

    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

    if HF_TOKEN:
        HF_TOKEN = HF_TOKEN.strip()

except Exception as e:
    print("Could not read HF_TOKEN from Kaggle Secrets:", repr(e))


# Optional environment-variable fallback.
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN")

    if HF_TOKEN:
        HF_TOKEN = HF_TOKEN.strip()


if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found.\n\n"
        "In Kaggle:\n"
        "1. Open Add-ons -> Secrets\n"
        "2. Create a secret named exactly HF_TOKEN\n"
        "3. Paste a NEW Hugging Face token as the value\n"
        "4. Enable/attach it to this notebook\n"
        "5. Make sure the same HF account has access to "
        "meta-llama/Llama-3.1-8B-Instruct"
    )

if not HF_TOKEN.startswith("hf_"):
    raise RuntimeError(
        "HF_TOKEN was found, but it does not look like a Hugging Face token."
    )

login(
    token=HF_TOKEN,
    add_to_git_credential=False,
)

api = HfApi(token=HF_TOKEN)

try:
    who = api.whoami()
    print("Authenticated Hugging Face account:", who.get("name", "unknown"))
except Exception as e:
    raise RuntimeError(
        "HF_TOKEN could not be authenticated.\n"
        f"Original error: {repr(e)}"
    )

try:
    info = api.model_info(
        MODEL_NAME,
        token=HF_TOKEN,
    )
    print("Model access verified:", info.id)
except Exception as e:
    raise RuntimeError(
        f"Your token works, but access to {MODEL_NAME} failed.\n"
        "Accept/request the Meta Llama license on the Hugging Face "
        "model page using the same account as this token.\n"
        f"Original error: {repr(e)}"
    )


# ============================================================
# 3. 4-BIT NF4 CONFIGURATION
# ============================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)


# ============================================================
# 4. TOKENIZER
# ============================================================

print("\nLoading tokenizer:", MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    use_fast=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"
tokenizer.truncation_side = "left"


# ============================================================
# 5. MODEL
# ============================================================

print("Loading model in 4-bit:", MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

model.eval()

device = model.get_input_embeddings().weight.device


# Llama 3.1 Instruct can terminate with EOS or <|eot_id|>.
TERMINATORS = [tokenizer.eos_token_id]

eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")

if (
    isinstance(eot_id, int)
    and eot_id >= 0
    and eot_id not in TERMINATORS
):
    TERMINATORS.append(eot_id)


print("\nModel loaded successfully.")
print("Input device:", device)
print("EOS terminators:", TERMINATORS)
print("Loaded in 4-bit:", getattr(model, "is_loaded_in_4bit", False))


# ============================================================
# 6. FEW-SHOT GENERATION
# ============================================================

def generate_answer(question, domain):

    if domain not in few_shot_examples:
        raise KeyError(
            f"No few-shot examples exist for domain: {domain}"
        )

    examples = few_shot_examples[domain][:NUM_SHOTS]

    messages = [
        {
            "role": "system",
            "content": (
                "তুমি বাংলাদেশের সরকারি সেবা বিষয়ক একজন সহকারী। "
                "দেওয়া উদাহরণগুলোর উত্তর দেওয়ার ধরন অনুসরণ করে "
                "নতুন প্রশ্নের সঠিক ও সংক্ষিপ্ত উত্তর বাংলায় দাও। "
                "প্রয়োজন হলে ফি, সময়, প্রয়োজনীয় কাগজপত্র ও "
                "প্রক্রিয়ার সঠিক তথ্য উল্লেখ করো। "
                "অপ্রয়োজনীয় ব্যাখ্যা দিও না।"
            ),
        }
    ]

    # Fixed 3-shot demonstrations.
    for ex in examples:

        messages.append({
            "role": "user",
            "content": f"প্রশ্ন: {ex['question']}",
        })

        messages.append({
            "role": "assistant",
            "content": ex["answer"],
        })

    # Actual test question.
    messages.append({
        "role": "user",
        "content": f"প্রশ্ন: {question}",
    })

    # Use Llama's own chat template and tokenize in one step.
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    )

    inputs = inputs.to(device)

    prompt_length = inputs["input_ids"].shape[1]

    with torch.inference_mode():

        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.05,
            eos_token_id=TERMINATORS,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
        )

    generated = output[0][prompt_length:]

    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True,
    ).strip()

    last_token = (
        generated[-1].item()
        if len(generated) > 0
        else None
    )

    truncated = (
        len(generated) >= MAX_NEW_TOKENS
        and last_token not in TERMINATORS
    )

    return answer, truncated


# ============================================================
# 7. QUICK ONE-QUESTION SANITY CHECK
# ============================================================

sample_row = tests.iloc[0]

sample_answer, sample_truncated = generate_answer(
    sample_row["question"],
    sample_row["domain"],
)

print("\nSanity-check question:")
print(sample_row["question"])
print("\nSanity-check prediction:")
print(sample_answer)
print("\nTruncated:", sample_truncated)


# ============================================================
# 8. RESUME SUPPORT
# ============================================================

PARTIAL = OUT / "predictions_partial.csv"

done = {}

if PARTIAL.exists():

    old = pd.read_csv(
        PARTIAL
    ).fillna("")

    for _, row in old.iterrows():

        done[
            (
                str(row["domain"]),
                str(row["id"]),
            )
        ] = row.to_dict()

print("\nAlready completed:", len(done))


# ============================================================
# 9. FULL INFERENCE
# ============================================================

predictions = []

for _, row in tqdm(
    tests.iterrows(),
    total=len(tests),
    desc="Llama-3.1-8B 3-Shot",
):

    key = (
        str(row["domain"]),
        str(row["id"]),
    )

    if key in done:

        result = done[key]

    else:

        answer, truncated = generate_answer(
            row["question"],
            row["domain"],
        )

        result = {
            "id": str(row["id"]),
            "domain": str(row["domain"]),
            "question": str(row["question"]),
            "gold": str(row["gold"]),
            "prediction": answer,
            "num_shots": NUM_SHOTS,
            "truncated": truncated,
        }

        done[key] = result

    predictions.append(result)

    # Checkpoint after every question.
    pd.DataFrame(
        predictions
    ).to_csv(
        PARTIAL,
        index=False,
        encoding="utf-8-sig",
    )


# ============================================================
# 10. SAVE FINAL PREDICTIONS
# ============================================================

pred_df = pd.DataFrame(predictions)

PREDICTIONS_PATH = OUT / "predictions.csv"

pred_df.to_csv(
    PREDICTIONS_PATH,
    index=False,
    encoding="utf-8-sig",
)

run_config = {
    "model": MODEL_NAME,
    "method": "few-shot",
    "shots": NUM_SHOTS,
    "example_selection": "fixed random examples per domain",
    "seed": 42,
    "max_input_tokens": MAX_INPUT_TOKENS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "do_sample": False,
    "quantization": "4-bit NF4",
    "test_questions": len(tests),
}

with open(
    OUT / "run_config.json",
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        run_config,
        f,
        ensure_ascii=False,
        indent=2,
    )


print("\nCompleted:", len(pred_df))

print(
    "Truncated:",
    pred_df["truncated"]
    .astype(str)
    .str.lower()
    .eq("true")
    .sum(),
)

print("Saved:", PREDICTIONS_PATH)


# Free the Llama model before CPU-heavy evaluation.
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()

print("GPU model memory released.")


GPU: Tesla T4
GPU memory: 14.56 GB
Authenticated Hugging Face account: arkaarkadutta
Model access verified: meta-llama/Llama-3.1-8B-Instruct

Loading tokenizer: meta-llama/Llama-3.1-8B-Instruct


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Loading model in 4-bit: meta-llama/Llama-3.1-8B-Instruct


...


Model loaded successfully.
Input device: cuda:0
EOS terminators: [128009]
Loaded in 4-bit: True


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(



Sanity-check question:
NID আবেদন করতে কী কী ডকুমেন্ট লাগে?

Sanity-check prediction:
আবেদনকারীর নাগরিকত্ব প্রমাণ (পাসপোর্ট, নাগরিকত্ব সনদ), জন্ম প্রমাণ (জন্ম সনদ, বেতনপত্র), বাসভিত্তিক প্রমাণ (ভৌগোলিক স্থান নির্ধারণ করার জন্য বাসভিত্তিক প্রমাণ), বাংলাদেশের নাগরিক হওয়ার প্রমাণ (পাসপোর্ট, নাগরিকত্ব সনদ) ইত্যাদি প্রয়োজন।

Truncated: False

Already completed: 0


Llama-3.1-8B 3-Shot:   0%|          | 0/248 [00:00<?, ?it/s]


Completed: 248
Truncated: 57
Saved: /kaggle/working/BanglaLLM/Llama/Few-Shot/predictions.csv
GPU model memory released.


In [23]:
# ============================================================
# CELL 3 — FEW-SHOT EVALUATION
# Exact Match, Fuzzy Match, Corpus BLEU,
# ROUGE-1/2/L, Token F1,
# BERT Precision, Recall and F1
# ============================================================

from collections import Counter
from rapidfuzz import fuzz
from sacrebleu.metrics import BLEU
from bert_score import score as bert_score


df = pd.read_csv(
    OUT / "predictions.csv"
).fillna("")

assert (
    df["gold"]
    .astype(str)
    .str.strip()
    .ne("")
    .all()
), "Some rows have an empty gold answer."


# ============================================================
# NORMALIZATION
# ============================================================

BN_TO_EN = str.maketrans(
    "০১২৩৪৫৬৭৮৯",
    "0123456789",
)


def normalize(text):

    text = unicodedata.normalize(
        "NFKC",
        str(text),
    )

    text = text.translate(
        BN_TO_EN
    ).lower()

    text = re.sub(
        r"[^\u0980-\u09FFA-Za-z0-9]+",
        " ",
        text,
    )

    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


def tokens(text):
    return normalize(text).split()


# ============================================================
# EXACT MATCH
# ============================================================

def exact_match(pred, gold):

    return float(
        normalize(pred)
        ==
        normalize(gold)
    )


# ============================================================
# TOKEN F1
# ============================================================

def token_f1(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p or not g:
        return 0.0

    overlap = sum(
        (
            Counter(p)
            &
            Counter(g)
        ).values()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / len(p)
    recall = overlap / len(g)

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ============================================================
# ROUGE-N
# ============================================================

def rouge_n(pred, gold, n):

    p = tokens(pred)
    g = tokens(gold)

    if len(p) < n or len(g) < n:
        return 0.0

    pn = Counter(
        tuple(p[i:i+n])
        for i in range(len(p)-n+1)
    )

    gn = Counter(
        tuple(g[i:i+n])
        for i in range(len(g)-n+1)
    )

    overlap = sum(
        (pn & gn).values()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / sum(pn.values())
    recall = overlap / sum(gn.values())

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ============================================================
# ROUGE-L
# ============================================================

def rouge_l(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p or not g:
        return 0.0

    dp = [0] * (len(g) + 1)

    for x in p:

        new = [0]

        for j, y in enumerate(g, 1):

            if x == y:
                new.append(
                    dp[j-1] + 1
                )
            else:
                new.append(
                    max(
                        dp[j],
                        new[-1],
                    )
                )

        dp = new

    lcs = dp[-1]

    precision = lcs / len(p)
    recall = lcs / len(g)

    if precision + recall == 0:
        return 0.0

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ============================================================
# ROW-LEVEL METRICS
# ============================================================

df["Exact Match"] = [
    exact_match(p, g)
    for p, g in zip(
        df["prediction"],
        df["gold"],
    )
]

df["Fuzzy Match"] = [
    fuzz.token_set_ratio(
        normalize(p),
        normalize(g),
    ) / 100
    for p, g in zip(
        df["prediction"],
        df["gold"],
    )
]

df["Token F1"] = [
    token_f1(p, g)
    for p, g in zip(
        df["prediction"],
        df["gold"],
    )
]

df["ROUGE-1"] = [
    rouge_n(p, g, 1)
    for p, g in zip(
        df["prediction"],
        df["gold"],
    )
]

df["ROUGE-2"] = [
    rouge_n(p, g, 2)
    for p, g in zip(
        df["prediction"],
        df["gold"],
    )
]

df["ROUGE-L"] = [
    rouge_l(p, g)
    for p, g in zip(
        df["prediction"],
        df["gold"],
    )
]


# ============================================================
# CORPUS BLEU
# ============================================================

bleu = BLEU(
    tokenize="none",
    smooth_method="exp",
    effective_order=True,
)

pred_bleu = [
    " ".join(tokens(x))
    for x in df["prediction"]
]

gold_bleu = [
    " ".join(tokens(x))
    for x in df["gold"]
]

corpus_bleu = (
    bleu.corpus_score(
        pred_bleu,
        [gold_bleu],
    ).score
    / 100
)


# ============================================================
# BERTSCORE
# ============================================================

bert_precision_mean = np.nan
bert_recall_mean = np.nan
bert_f1_mean = np.nan

try:
    print("Calculating BERTScore...")

    P, R, F1 = bert_score(
        df["prediction"].astype(str).tolist(),
        df["gold"].astype(str).tolist(),
        model_type="bert-base-multilingual-cased",
        batch_size=4,
        device="cpu",
        verbose=True,
        idf=False,
        rescale_with_baseline=False,
    )

    df["BERT Precision"] = P.cpu().numpy()
    df["BERT Recall"] = R.cpu().numpy()
    df["BERT F1"] = F1.cpu().numpy()

    bert_precision_mean = float(df["BERT Precision"].mean())
    bert_recall_mean = float(df["BERT Recall"].mean())
    bert_f1_mean = float(df["BERT F1"].mean())

except Exception as e:

    print("\nBERTScore failed:")
    print(repr(e))
    print(
        "The remaining metrics will still be saved. "
        "BERTScore values will be NaN."
    )

    df["BERT Precision"] = np.nan
    df["BERT Recall"] = np.nan
    df["BERT F1"] = np.nan


# ============================================================
# FINAL RESULT
# ============================================================

result = pd.DataFrame({

    "metric": [
        "Exact Match",
        "Fuzzy Match",
        "Corpus BLEU",
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
        "Token F1",
        "BERT Precision",
        "BERT Recall",
        "BERT F1",
    ],

    "score": [
        float(df["Exact Match"].mean()),
        float(df["Fuzzy Match"].mean()),
        float(corpus_bleu),
        float(df["ROUGE-1"].mean()),
        float(df["ROUGE-2"].mean()),
        float(df["ROUGE-L"].mean()),
        float(df["Token F1"].mean()),
        bert_precision_mean,
        bert_recall_mean,
        bert_f1_mean,
    ],
})


# ============================================================
# SAVE
# ============================================================

df.to_csv(
    OUT / "predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

result.to_csv(
    OUT / "result.csv",
    index=False,
    encoding="utf-8-sig",
)

display(result)

print("\nSaved:")
print(RAG_DIR / "unified_train.json")
print(OUT / "few_shot_examples.json")
print(OUT / "run_config.json")
print(OUT / "predictions_partial.csv")
print(OUT / "predictions.csv")
print(OUT / "result.csv")


Calculating BERTScore...


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

computing greedy matching.


  0%|          | 0/62 [00:00<?, ?it/s]

done in 27.83 seconds, 8.91 sentences/sec


,metric,score
0,Exact Match,0.000000
1,Fuzzy Match,0.572658
2,Corpus BLEU,0.048763
3,ROUGE-1,0.262622
4,ROUGE-2,0.115516
5,ROUGE-L,0.231699
6,Token F1,0.262622
7,BERT Precision,0.723804
8,BERT Recall,0.735341
9,BERT F1,0.728328



Saved:
/kaggle/working/BanglaLLM/RAG/unified_train.json
/kaggle/working/BanglaLLM/Llama/Few-Shot/few_shot_examples.json
/kaggle/working/BanglaLLM/Llama/Few-Shot/run_config.json
/kaggle/working/BanglaLLM/Llama/Few-Shot/predictions_partial.csv
/kaggle/working/BanglaLLM/Llama/Few-Shot/predictions.csv
/kaggle/working/BanglaLLM/Llama/Few-Shot/result.csv
